In [1]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = "https://api.deepseek.com/v1"

api_key

'sk-ebfb06c810a1412fa88b260afe7d5d46'

In [2]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",    
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
llm

ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x7a2ed2b63ad0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7a2ed27507a0>, root_client=<openai.OpenAI object at 0x7a2ed2bffbc0>, root_async_client=<openai.AsyncOpenAI object at 0x7a2ed2bffb60>, model_name='deepseek-chat', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), max_retries=2, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1')

In [5]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import Runtime, before_model

# @before_model   
# def before_model_print(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
#     print(f" state: {state}")
#     print(f" runtime: {runtime}")
#     messages = state["messages"]
#     context = runtime.context
#     print(f" messages: {messages}")
#     print(f" context: {context}")
#     return {
#         "messages": state["messages"]
#     }

from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def query_word_list_by_user(user_id: int) -> str:
    """Query word list for a given user."""
    return f"Word list for user {user_id}"



agent = create_agent(
    model=llm,
    tools=[get_weather],
    # middleware=[before_model_print],
    system_prompt="You are a helpful assistant",
)
human_message = HumanMessage(content="What's the weather in New York?")
response = agent.stream({
    "messages": [human_message]
},stream_mode="values")
# define message as a list of dict[str,any]
messages: list[dict[str, Any]] = []
for msg in response:
    messages.append(msg)


In [ ]:

# for msg in messages:
#     print("Human:", msg)

from langchain_core.messages import ToolMessage
from langchain_core.messages import AIMessage
last_message = messages[-1]['messages']
for msg in last_message:
    # print(type(msg))
    if (type(msg) == HumanMessage):
        print("Human:", msg.content)
    elif (type(msg) == AIMessage):
        print("AI:", msg.content)
    elif (type(msg) == ToolMessage):
        print("Tool:", msg.content)
    
# last_message

Human: What's the weather in New York?
AI: I'll check the weather in New York for you.
Tool: It's always sunny in New York!
AI: According to the weather service, it's always sunny in New York!
